# Chapter 6
In this chapter, we explore **Multivariate Pattern Analysis (MVPA)** to decode brain states from *MEG/EEG signals*. We will move beyond *single-sensor analysis* and use machine learning to classify experimental conditions (e.g., Left vs. Right Audio). Specifically, we will cover:
+ **Spatio-Temporal Decoding:** Classifying entire trials to see if conditions differ.
+ **Temporal Decoding (Sliding Estimator):** Training classifiers at each time point to reveal when the brain processes differ.
+ **Pipeline Construction:** Using mne.decoding and scikit-learn to build robust, cross-validated decoding pipelines.

## Libraries and Config

In [ ]:
 # this ensures that plots open in a new window
%matplotlib qt

In [ ]:
import pathlib
import matplotlib
import matplotlib.pyplot as plt
import mne
import numpy as np

matplotlib.use("QtAgg")
mne.set_log_level("WARNING")

## Loading the pre-processed epochs

For temporal decoding (and machine learning in general), it’s important to start from data that has been consistently preprocessed, because models will otherwise learn systematic noise and artifacts instead of task-related brain signals (“garbage in, garbage out”). Since the epochs were already cleaned in the previous chapter (e.g., with ICA), they can be loaded directly as an MNE Epochs object using mne.read_epochs():

In [ ]:
pp_epochs = mne.read_epochs(
    pathlib.Path("out_data") / "epochs-ica-cleaned-epo.fif"
)

pp_epochs

Now let's only focus on the *'Auditory'* event and more specifically  the difference between *'Auditory/Left'* and *'Auditory/Right'* epochs:

In [ ]:
epochs_auditory = pp_epochs['Auditory']

epochs_auditory

## Calculate Empirical Evoked Difference

Now before starting with decoding the *'Auditory/Left'* and *'Auditory/Right'* let us first calculate **Empirical Evoked Difference**. We calculate it for two main reasons:
1. **To Confirm a Signal Exists:** Before asking a maching learning model to distinguish between "Auditory/Left" and "Auditory/Right", you want to verify that there is actually a measurable difference between them in the brain data. If the "Empirical Difference" (the simple subtraction of the two conditions' ERPs) is a flat line, the decoder will likely fail because there is nothing to find.

2. **To Guide Interpretation:** It shows you when and where the difference are strongest (e.g., "The left ear signal is stronger at 100ms in the right hemisphere"). This helps you validate the decoder's results later. If the decoder says "I found a difference at 500ms" but your empirical difference shows the activiy is at 100ms, you might have a bug or an artifact.

So let's first calculate the evoked responses for both events' epochs:

In [ ]:
evoked_aud_left = epochs_auditory['Left'].average()
evoked_aud_right = epochs_auditory['Right'].average()

Now we will combine the evoked using the `mne.combine_evoked()` function, where we will pass both of the evoked objects and  `weights=[1, -1]` which basically means `evoked[0]*1 + evoked[1]*-1` (i.e., the differnce between them): 

In [ ]:
auditory_evoked_diff = mne.combine_evoked(
    all_evoked=[evoked_aud_left, evoked_aud_right],
    weights=[1, -1] # (evoked_aud_left * 1) + (evoked_aud_right * -1)
)

Let's visualise a (Global Field Power) plot of this difference we calculated:

In [ ]:
auditory_evoked_diff.plot(gfp=True, spatial_colors=True)

Let's also compare th GFPs of the evoked objects and their differnce object using `mne.viz.plot_compare_evokeds()` function:

In [ ]:
mne.viz.plot_compare_evokeds(
    evokeds={
        'Left Auditory': evoked_aud_left,
        'Right Auditory': evoked_aud_right,
        'Difference (Left - Right)': auditory_evoked_diff
    }
)

Fantastic, we can clearly see differences in the evoked response after the trigger. However, before we build the decoder, notice that we have an uneven number of epochs for each condition — 'Auditory/Left': 72 vs 'Auditory/Right': 73.
<br/>
This class imbalance is critical because, depending on our evaluation metric (e.g., accuracy), a model could achieve "better than chance" performance simply by guessing the more frequent class. While we could use metrics like AUC or F1-score to handle this, the cleanest approach for a tutorial is often to remove this bias at the source. Luckily, the Epochs object includes a method called .equalize_event_counts(), which automatically drops trials to ensure exactly equal numbers for every condition:

> **Pro tips:** While `.equalize_event_counts()` is convenient, it works by throwing away real data (undersampling) to balance the classes. In a real research pipeline where every trial is precious, a better alternative is often Stratified Cross-Validation. This technique keeps all your data but ensures that every "fold" (training and testing split) maintains the same ratio of classes as the original dataset. This prevents bias without sacrificing data quantity.

In [ ]:
epochs_auditory.equalize_event_counts(
    event_ids=epochs_auditory.event_id
)

epochs_auditory

## Multivariate Statistics (Decoding / MVPA) Supervised Learning on MEG/EEG Data
The goal of this section is to train a supervised binary classifier (decoder). The classifier will take a data matrix `X` as input and learn to predict a target vector `y` (labels *0* and *1*).
<br />
<br />
Here, `X` represents our neural data. Unlike traditional univariate analysis which looks at one sensor at a time, we will use a multivariate pattern that combines all sensors and all time points within each epoch, so each trial becomes one feature vector (channels × time) and treat it as the data. By feeding the classifier the full Spatio-temporal pattern of a sensor values (e.g., in fT/cm), we enable it to learn subtle, distributed differences between `'Auditory/Left'` and `'Auditory/Right'` trials that single sensors might miss.

Here, `X` represents our neural data. Unlike traditional univariate analysis (one sensor at a time), we use a multivariate pattern that combines all sensors and all time points within each epoch, so each epoch (trial) becomes one feature vector with **channels × time** features. By giving the classifier this full **spatio-temporal pattern** (e.g., gradiometer values in fT/cm), it can learn subtle, distributed differences between `'Auditory/Left'` and `'Auditory/Right'` trials that may not be visible in any single sensor alone.
<br />
<br />
Let's start by creating the target labels `y`:

> Pro tip: Types of Decoding in M/EEG:
>   + **Spatio-temporal decoding (whole-epoch-at-once):** Use the entire epoch (*all sensors × all times*) as features to make one prediction per trial; often gives strong performance, but doesn’t directly tell you when the info occurred unless you add extra analyses.
>   + **Temporal decoding / sliding estimator (when-info):** Repeat spatial decoding at every time point (100 ms, 105 ms, 110 ms, …) to get an accuracy curve over time; tells you when the information becomes decodable.
>   + **Spatial decoding (one instant):** Pick one time point (say 100 ms after stimulus) and use the values from all sensors at that instant to predict the label; one model per chosen time point, features = channels.
>   + **Temporal generalization (stability-over-time):** Train a model at one time point (e.g., 100 ms) and test it at other time points (e.g., 200 ms); if it still works later, the brain pattern is stable, and if not, the representation likely changes over time.





In [ ]:
# create an empty array of size = number of trials 
y = np.empty(len(epochs_auditory.events), dtype=int)

# crate a boolean mask for 'Left' trials
mask_left = epochs_auditory.events[:, 2] == epochs_auditory.event_id["Auditory/Left"]
# create a boolean mask for 'Right' trials
mask_right = epochs_auditory.events[:, 2] == epochs_auditory.event_id["Auditory/Right"]

# encode 'Left' trials as 0 and 'Right' trials as 1
y[mask_left] = 0
y[mask_right] = 1

print(f"Size of y: {y.size}\n{y}")

Now, let's create the input matrix `X`.
<br />
<br />
&emsp;&emsp;For this classification task, we will focus only on the gradiometer channels. We can extract this specific data by first calling `.pick_types(meg="grad")` on our epochs, and then retrieving the values with `.get_data()`:

> Note: If you wanted to use magnetometer channels, you would pass `meg='mag'`. For EEG channels, you would use `meg=False, eeg=True` inside `.pick_types()`.

In [ ]:
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg="grad")

# retrive the data as a 3D numpy array 
# The array has the shape: (n_trails, n_channels, n_timepoints)
X = epochs_auditory_grad.get_data() #
print(f"Shape of X: {X.shape}")

From the output, you can see we have **144 trials/epochs** (the total of *'Auditory/Left'* and *'Auditory/Right'* events), readings from **203 different channels**, and **481 sampling time points** for each trail/epoch.
<br />
<br />
To convert this into a single **spatio-temporal feature vector** for each trial, we need to flatten the "channel" and "time" dimensions together. The goal is to turn our 3D array `(n_trials, n_channels, n_timepoints)` into a 2D matrix `(n_trials, n_features)`.
<br />
<br />
Luckily, we can use the numpy `.reshape()` function to flatten the last two axes automatically:

In [ ]:
n_trials = X.shape[0]
X = X.reshape(n_trials, -1) # The '-1' tells numpy to combine the remaining dimensions
   
print(f"New shape of X: {X.shape}")

### Binary Classifier

We will use `scikit-learn` to build our classification model. Best practice is to use a **Pipeline**, which bundles the preprocessing and modeling steps together into a single object.

To create this pipeline, we use the `make_pipeline()` function from `sklearn.pipeline` and pass it two steps:

1. __Scaling (`StandardScaler`):__ <br />
The raw MEG/EEG data values are extremely small (often $< 10^{-12}$). Many machine learning algorithms struggle with such tiny numbers or variables with vastly different ranges. We use `StandardScaler()` from `sklearn.preprocessing` to standardize the data, ensuring every feature has a mean of 0 and a standard deviation of 1.

2. __The Classifier (`LogisticRegression`):__
This is the actual model that will learn to distinguish between the conditions. We will use `LogisticRegression()` from `sklearn.linear_model`, which is a simple, linear classifier that works surprisingly well for high-dimensional neuroimaging data.

> __Pro Tip: Why Linear Models?__<br />
> You might wonder why we use a simple Logistic Regression instead of complex Deep Learning or Random Forests. In M/EEG decoding, we often have fewer trials than features (e.g., 144 trials vs. 97,000 features).
>
> + Complex models (like Neural Networks) tend to overfit massively on this kind of "wide" data (memorizing the noise instead of the signal).
>
> + Linear models (like Logistic Regression or SVM) are robust, faster to train, and—crucially—allow us to inspect the weights later to see which sensors drove the decision (interpretability).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

2. __Cross-Validation Strategy:__ <br />
To evaluate our model's performance reliably, we cannot just train it once. We need to use **Cross-Validation**, where the data is split multiple times into different "Training" and "Testing" sets.
<br />
We will use `StratifiedKFold` from `sklearn.model_selection`. This is smarter than a standard random split because it enforces ___Stratification__: it ensures that every fold preserves the original percentage of samples for each class (e.g., 50% Left / 50% Right)_. This prevents a situation where a random split accidentally puts all the "Left" trials in the test set, which would ruin the evaluation.

>__Pro Tip: To Shuffle or Not to Shuffle?__<br />
We set `shuffle=True` here, which is standard for *Event-Related Potentials (ERPs)* where trials are assumed to be independent events (e.g., Trial 10 doesn't depend on Trial 9).
> <br />
> <br />
> However, if you are analyzing continuous data, resting-state data, or specific tasks where the subject's state drifts slowly over time (e.g., fatigue or learning effects), shuffling can cause Data Leakage. In those cases, the model might "cheat" by using the temporal proximity of trials rather than the brain signal itself. For such data, use `KFold(shuffle=False)` or keeping blocks of time together is safer.

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 5  # We will split data into 5 parts: 4 for training, 1 for testing, repeated 5 times.

# Create the cross-validator object
cross_validator = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,     # Shuffle data before splitting to break any order-dependency
    random_state=42   # Fix the seed for reproducibility
)

3. **Training and Evaluation:** <br />
To train and evaluate the model in one go, we use the `cross_val_score()` function. This utility automates the entire loop: it splits the data according to our `cross_validator` object, trains the `estimator` (classifier) on the training portion, and evaluates it on the test portion using the specified `scoring` metric.
<br />
It returns an array of scores (one for each fold). This gives us a robust estimate of how well the model generalizes to new data.

> ___Note:__ `cross_val_score` does not return a trained model_. It creates temporary copies of the classifier for each fold and discards them after scoring. If you need a trained model for later (e.g., to predict on new data), you must call `classifier.fit(X, y)` separately on the full dataset after this step.

> __Pro Tips:__ <br />
>    + Setting `n_jobs=-1` is a great time-saver as it runs the 5 folds in parallel on different CPU cores. However, for very small datasets, the overhead of creating parallel jobs might actually make it slower than `n_jobs=1`. For M/EEG data, usually `-1` is the right choice.
>   + For `scoring=` we will use `'roc_auc'`; **ROC AUC** stands for **“Area Under the Receiver Operating Characteristic Curve”**, and it measures how well your classifier can separate the two classes across all possible decision thresholds. The **ROC curve** plots the true positive rate (how many “Auditory/Right” trials you correctly detect) against the false positive rate (how many “Auditory/Left” trials you incorrectly label as “Right”) as you slide the threshold from strict to lenient. The **AUC** is the area under that curve: `1.0` means perfect separation, `0.5` means chance-level (like random guessing), and values in between reflect how often the model ranks a true “Right” trial higher than a true “Left” trial, and for `AUC < 0.5` - the model is predicting the opposite class systematically (which usually means a label flipping bug!). **ROC AUC** is popular in decoding because it is less sensitive to class imbalance than plain accuracy and doesn’t depend on picking one arbitrary threshold.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the scoring metric
scoring_metric = "roc_auc" # Area Under the Receiver Operating Characteristic Curve

# Perform training and evaluation
# This runs the 5-fold cross-validation and returns 5 scores
scores = cross_val_score(
    estimator=classifier,
    X=X,
    y=y,
    cv=cross_validator,
    scoring=scoring_metric,
    n_jobs=-1 # Use all available CPU cores for speed
)

# Calculate Mean and Standard Deviation of the scores
mean_score = round(np.mean(scores), 3)
std_score = round(np.std(scores), 3)

print(f"ROC AUC scores for each fold: {scores}")
print(f"Mean ROC AUC: {mean_score} ± {std_score}")

It is often better to visualize the spread of cross-validation scores rather than just looking at the single mean number. A **Box Plot** is perfect for this - it shows us not just the average performance, but also how consistent (stable) the model is across different folds.

> __Note: How to Read this Box Plot:__
> + _The Box:_ Represents the middle 50% of the scores (from the 25th to 75th percentile). A short box means the model performs very consistently.
>
> + _The Orange Line:_ The Median score (the middle value).
>
> + _The Green Triangle:_ The Mean (average) score.
>
> + _The Whiskers:_ The vertical lines show the full range, extending down to the Minimum score (worst fold) and up to the Maximum score (best fold).
>
> What to look for: Ideally, you want a box that is high up (good accuracy) and very short (consistent performance). If the box is huge, your model might be unstable (working great on some data but failing on others).

In [ ]:
fig, ax = plt.subplots()
ax.boxplot(
    scores,
    showmeans=True,  # Show the mean score as a green triangle
    whis=(0, 100),   # Extend whiskers to show the full range (Min to Max)
    labels=['Left vs Right Auditory']
)
ax.set_ylabel('ROC AUC Score')
ax.set_title('Classifier Performance Across Cross-Validation Folds')
plt.show()

### Using `mne.decoding` module

So far, we have implemented many steps manually (like reshaping the data or creating the scaler). Luckily, MNE provides the `mne.decoding` module, which contains specialized classes designed to integrate *M/EEG* data directly with the **scikit-learn** ecosystem. This makes our code cleaner and less error-prone.

1. __Prepare the Data (X and y):__ <br />
We start by extracting the data again. Note that we don't need to manually reshape `X` or convert `y` into 0/1 labels right now; the MNE decoding tools are smart enough to handle the raw 3D array and the original event codes (e.g., 100 vs 200):

> __Pro Tip: Why use `mne.decoding`?__
> Standard *scikit-learn* expects 2D data (samples × features). M/EEG data is 3D (samples × channels × time).
> <br />
> The classes in `mne.decoding` (like `Scaler`, `Vectorizer`, `SlidingEstimator`) act as bridges. They accept 3D MNE data, perform the necessary reshaping or sliding-window operations internally, and then pass standard 2D data to scikit-learn. This saves you from having to write complex `reshape()` or loop logic yourself!

In [ ]:
# Pick the gradiometer channels
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg="grad")

# X: Retrieve the data as a 3D numpy array (n_epochs, n_channels, n_times)
X = epochs_auditory_grad.get_data()

# y: Retrieve the Event Codes directly (from the 3rd column of the events array)
y = epochs_auditory_grad.events[:, 2]

2. __Create the Pipeline:__ <br />
We will build the classification pipeline again using `make_pipeline`. This time, instead of using the generic scikit-learn transformers, we will use specialized classes from `mne.decoding`:
+ `Scaler`: Unlike the standard scaler, MNE's `Scaler` understands neuroimaging data. By passing it the `epochs.info`, it knows which channels are _Gradiometers vs Magnetometers vs EEG_ and scales them appropriately (separately) based on their physical units.

+ `Vectorizer`: This replaces our manual `.reshape()` step. It automatically flattens the last dimensions of the 3D data (Channels × Time) into a single feature vector for the classifier.

> __Pro Tip: Why not just use `StandardScaler`?__ <br />
> If you mixed Magnetometers (measured in Tesla, $\approx 10^{-15}$) and EEG (measured in Volts, $\approx 10^{-6}$) and fed them to a standard `StandardScaler`, it might treat them as a single group or struggle with the massive scale difference.
>
> `mne.decoding.Scaler` is smarter: it splits the data by channel type (e.g., scales all MAGs together, all EEGs together) and then combines them. This ensures that one sensor type doesn't dominate the classifier just because its physical units are larger!

In [ ]:
from mne.decoding import Scaler, Vectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Create the Classifier pipeline
classifier = make_pipeline(
    # MNE Scaler needs 'info' to correctly handle different sensor types (e.g., MAG vs GRAD)
    Scaler(info=epochs_auditory_grad.info),
    
    # MNE Vectorizer automatically flattens the 3D data into 2D (n_samples, n_features)
    Vectorizer(),
    
    # Standard Scikit-Learn Classifier
    LogisticRegression()
)

3. __Training and Evaluation:__ <br />
To train and evaluate the model, we use `cross_val_multiscore()` from `mne.decoding`.
<br />
<br />
This function is very similar to scikit-learn's `cross_val_score`, but it is optimized to work with MNE estimators. Crucially, for "Temporal Decoding" task(decoding each time point separately), as `cross_val_multiscore` can handle returning multiple scores (one per time point) automatically, whereas the standard sklearn function might struggle with the dimensions.
<br />
<br />
We can pass a simple integer (e.g., `5`) to `cv`, and it will default to a K-Fold split. However, passing a `cross validator` object like the `StratifiedKFold` object (from the previous section) is often safer to ensure balanced classes.

> __Pro Tip: Why `cross_val_multiscore`?__ <br />
> If you are just doing **"Spatio-Temporal" decoding** (one score per trial), sklearn.`cross_val_score` works fine.
<br />
<br />
> However, `mne.cross_val_multiscore` shines for "Temporal Decoding" task (decoding each time point separately), as `cross_val_multiscore` can handle returning multiple scores (one per time point) automatically, whereas the standard sklearn function might struggle with the dimensions. 
<br />
> For example, if you use a **Sliding Estimator** (decoding time-point by time-point), the result isn't just one number - it's a whole time-series of accuracy. `cross_val_multiscore` automatically preserves this shape `(n_folds, n_times)`, making it the standard choice for MNE pipelines.

In [ ]:
from mne.decoding import cross_val_multiscore

# We can use the same cross-validator object we defined earlier, or just an integer (e.g., 5)
n_splits = 5
scoring_metric = "roc_auc"

scores = cross_val_multiscore(
    estimator=classifier,
    X=X,
    y=y,
    cv=n_splits,  # Or pass a 'cross_validator' object
    scoring=scoring_metric,
    n_jobs=-1     # Use all cores
)

# Calculate Mean and Standard Deviation
roc_auc_mean = round(np.mean(scores), 3)
roc_auc_std = round(np.std(scores), 3)

print(f"ROC AUC scores for each fold: {scores}")
print(f"Mean ROC AUC: {roc_auc_mean} ± {roc_auc_std}")

## Temporal Decoding / Sliding Estimator
In the previous examples, we trained a classifier to discriminate between experimental conditions using the *spatio-temporal patterns* of **entire trials (Spatio-Temporal Decoding)**. While this tells us that the conditions are different, it doesn't tell us when the difference happens.
<br />
<br />
An interesting neuroscientific question is: **Exactly when do the brain signals for two conditions differ?**
<br />
<br />
We can answer this by fitting a separate classifier at **every single time point.**
+ If the classifier works well at $t=100ms$, we know the brain patterns differed at that specific instant.
+ If it fails at $t=500ms$, we know the brain signals were indistinguishable then.

By tracking classification performance over the entire epoch, we can pinpoint **precisely at which time** *the spatial activation patterns start to diverge between the two events*. This technique of decoding over time is known as **Temporal Decoding** (or using a **Sliding Estimator**).

1. **Prepare the Data (X and y):**
Just like before, we start by extracting the data (`X`) and the labels (`y`):

In [ ]:
# Pick only the gradiometer channels
epochs_auditory_grad = epochs_auditory.copy().pick_types(meg="grad")

# X: (n_epochs, n_channels, n_times)
X = epochs_auditory_grad.get_data()

# y: Event codes
y = epochs_auditory_grad.events[:, 2]


2. **Pre-processing (Scaling):** <br />
Before training, we need to scale our data. While we could put a scaler inside the pipeline, it is often more efficient to scale the entire 3D data array (`X`) once globally using MNE's `Scaler`. This ensures all channel types (e.g., Gradiometers) are scaled correctly based on their physical units before we slice them up for temporal decoding.

> **Pro Tip: Why Scale Globally?** <br />
You might be tempted to put the `Scaler` inside a `make_pipeline` with the classifier (like we did in the previous section). However, that can cause issues in **Temporal Decoding**:
>
> 1. The MNE `Scaler` Error: If you put `mne.decoding.Scaler` inside a `SlidingEstimator`, it will crash. The `Scaler` expects **3D** data (`Epochs` structure), but the Sliding Estimator slices the data into **2D** chunks (`Time Step`) before passing it down. The Scaler sees the wrong dimensions and throws a `ValueError`.
>
> 2. The `StandardScaler` Risk: You could use **Scikit-Learn's** `StandardScaler` inside the pipeline (since it handles 2D data fine). However, `StandardScaler` is *"blind"* to channel types. If you had mixed sensors (e.g., *MAG + GRAD*), it would scale them all together, potentially distorting the data.
>
> *Solution: Scaling globally first gives us the best of both worlds: MNE's smart channel-aware scaling and compatibility with the Sliding Estimator.*

In [ ]:
from mne.decoding import Scaler

# Initialize MNE's Scaler with the info object (so it knows channel types)
scaler = Scaler(epochs_auditory_grad.info)

# Scale the entire 3D data array (n_epochs, n_channels, n_times)
X_scaled = scaler.fit_transform(X)

2. **Defining Base Estimator:** <br />
We need to define the *"Base Estimator"* — the model that will be trained at each individual time point. <br /> <br />
Since the `SlidingEstimator` (which we will use in the next step) automatically slices the data into **2D** chunks `(n_epochs × n_channels)` for each time point, we don't need a Vectorizer anymore. The input to our classifier at any single millisecond is just a spatial vector.<br /> <br />
Because we have already scaled the data globally, our model doesn't need any preprocessing steps attached to it. Therefore, instead of creating a pipeline with just one step, it is cleaner to just instantiate the classifier (`LogisticRegression`) directly.

> **Note: Where did the Vectorizer go?** <br />
> In **Spatio-Temporal decoding**, we had to flatten Time and Channels together, so we needed `Vectorizer`.
> 
> In **Temporal (Sliding) decoding**, we look at one time point at a time. At $t=100ms$, our data is just a snapshot of sensors. It is already a simple 2D matrix `(n_epochs, n_channels)`, so the classifier can consume it directly without reshaping.

In [ ]:
from sklearn.linear_model import LogisticRegression

# 1. Define the "base" estimator (what happens at each time point)
# Note: We do NOT need Vectorizer (data is 2D at each step) or Scaler (data is already scaled).
base_classifier = LogisticRegression(solver='liblinear', random_state=42)


3. __Defining Sliding Estimator:__ <br />
The Sliding Estimator acts as a "meta-model": **it will take our base classifier, copy it `n_timepoints` times, and train one instance for every single time point in our data.** We use the `SlidingEstimator` class from `mne.decoding`:

In [ ]:
from mne.decoding import SlidingEstimator

temporal_decoder = SlidingEstimator(
    base_estimator=base_classifier,  # The LogisticRegression we just defined
    scoring="roc_auc",               # Use ROC AUC as the scoring metric
    n_jobs=-1,                       # Parallelize across time points (faster!)
    verbose=True                     # Enable output to monitor progress
)

4. __Training and Evaluation:__ <br />
We will use `cross_val_multiscore()` to run the full cross-validation loop.
Crucially, because we are using a `SlidingEstimator`, this function will return a **2D array** of scores with shape `(n_folds, n_time_points)`, giving us the accuracy for every fold at every millisecond.

In [ ]:
from mne.decoding import cross_val_multiscore
import numpy as np

# Run cross-validation
# Note: We pass X_scaled (not raw X) because we scaled it globally in Step 2!
scores = cross_val_multiscore(
    estimator=temporal_decoder,
    X=X_scaled,
    y=y,
    cv=5,        # 5-fold cross-validation
    n_jobs=-1    # Use all CPU cores
)

print(f"Scores shape: {scores.shape}") # Should be (5, 481)

# Calculate Mean accuracy across the 5 folds for each time point
# axis=0 averages the folds, leaving us with one score per time point
mean_scores = np.mean(scores, axis=0)
print(f"Mean scores shape: {mean_scores.shape}") # Should be (481,)

5. **Plotting time score:** <br />
Since temporal decoding gives us a separate score for every millisecond, averaging them into a single number would hide the interesting dynamics. Instead, we plot the time course of the classification performance:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(0.5, color="k", linestyle="--", label="Chance (0.5)") # Chance level/Random Guess for binary classification
ax.axvline(0, color="k", linestyle="--")  # Mark stimulus onset
ax.plot(
    epochs_auditory_grad.times,
    mean_scores,
    label="Temporal Decoding AUC",
    linewidth=2
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Mean ROC AUC")
ax.legend()
ax.set_title("Temporal Decoding of Left vs Right Auditory Stimuli")
fig.suptitle("Sensor Space Decoding Results", fontsize=16)
plt.show()


6. **Results Interpretation:**
<div style="text-align: center;">
<img src="imgs/temporal_decoding_res.png" alt="Temporal Decoding mean score across all time point results" width="500">
</div>


Looking at the plot, we can observe clear temporal dynamics:

+ **Baseline Period (-0.2s to 0s)**: The decoding performance oscillates around **0.5 (chance level/random guessing)**. This is a crucial sanity check—it confirms that before the sound was played, the classifier could not predict the trial type (no "data leakage").

+ **Stimulus Onset (> 0s):** Shortly after $t=0$, we see a sharp rise in accuracy. The mean ROC AUC peaks significantly above `0.9`, indicating that the brain patterns for "Left" vs "Right" sounds are distinct and highly classifiable at these time points.

+ **Return to Baseline:** After the initial response (around 300-400ms), the performance drops back towards chance level.

This confirms our earlier *"Empirical Evoked Difference"* check: there is a real, strong difference between the two conditions, and this decoder has successfully pinpointed **exactly when that difference occurs (peaking around ~100ms)**.

# The End & Have a great day!